# EV3 — Análisis Exploratorio de Datos (EDA)
### Sistema de Recomendación de Hardware para PC

Este notebook cruza tres fuentes de datos para generar recomendaciones de hardware:
- **CSV (ETL):** Requisitos de juegos y popularidad de hardware en Steam
- **MySQL (BD):** Catálogo de componentes, tiers y builds pre-armadas
- **API eBay:** Precios reales de componentes en CLP

## 1. Importaciones y Conexiones

In [171]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import mysql.connector
import os
import warnings
warnings.filterwarnings('ignore')

# Rutas relativas desde la carpeta /eda
BASE = os.path.join(os.path.dirname(os.getcwd()), '') if os.path.basename(os.getcwd()) == 'eda' else ''
KAGGLE_CSV      = os.path.join(BASE, 'data', 'kaggle', 'games_sample_15.csv')
STEAM_CSV       = os.path.join(BASE, 'data', 'steamhwsurvey', 'steam_sample_15.csv')
BUILDS_CSV      = os.path.join(BASE, 'data', 'builds_populares.csv')
GAMES_PRICED    = os.path.join(BASE, 'data', 'kaggle', 'games_sample_15_PRICED.csv')
STEAM_PRICED    = os.path.join(BASE, 'data', 'steamhwsurvey', 'steam_sample_15_PRICED.csv')

print('Rutas configuradas correctamente.')

Rutas configuradas correctamente.


In [172]:
# Conexión a MySQL
try:
    conn = mysql.connector.connect(
        host='localhost',
        port=3306,
        user='root',
        password='',
        database='tienda_hardware_intelligence'
    )
    print('Conexión a MySQL exitosa.')
except Exception as e:
    print(f'Error de conexión: {e}')
    conn = None

Conexión a MySQL exitosa.


## 2. Carga de Datos

In [173]:
# --- FUENTE 1: CSV (ETL) ---
df_games  = pd.read_csv(KAGGLE_CSV)
df_steam  = pd.read_csv(STEAM_CSV)
df_builds = pd.read_csv(BUILDS_CSV)

# CSV con precios inyectados por la API de eBay (generados por fetch_prices.py)
df_games_priced = pd.read_csv(GAMES_PRICED) if os.path.exists(GAMES_PRICED) else df_games.copy()
df_steam_priced = pd.read_csv(STEAM_PRICED) if os.path.exists(STEAM_PRICED) else df_steam.copy()

print(f'Juegos cargados: {len(df_games)}')
print(f'Datos Steam cargados: {len(df_steam)}')
print(f'Builds cargadas: {len(df_builds)}')

Juegos cargados: 15
Datos Steam cargados: 20
Builds cargadas: 4


In [174]:
# --- FUENTE 2: MySQL (BD) ---
if conn:
    df_components = pd.read_sql('SELECT c.id, c.name, c.categoria, t.tier_name FROM component c JOIN component_tiers t ON c.component_tiers_id = t.id', conn)
    df_prices_db  = pd.read_sql('SELECT c.name as componente, m.price_clp FROM market_prices_external m JOIN component c ON m.component_id = c.id', conn)
    df_builds_db  = pd.read_sql('SELECT bt.template_name, c.name as componente, c.categoria, t.tier_name FROM build_templates bt JOIN build_components bc ON bt.id = bc.build_templates_id JOIN component c ON bc.component_id = c.id JOIN component_tiers t ON c.component_tiers_id = t.id', conn)
    df_steam_db   = pd.read_sql('SELECT c.name as componente, s.global_share_percentage as porcentaje FROM steam_hardware_survey s JOIN component c ON s.component_id = c.id', conn)
    df_games_db   = pd.read_sql('SELECT g.titulo as juego, c.name as componente, c.categoria, gr.requirement_type FROM games g JOIN game_requeriments gr ON g.id = gr.games_id JOIN component c ON gr.component_id = c.id', conn)
    print('Tablas MySQL cargadas correctamente.')
else:
    print('Sin conexión MySQL. Usando solo CSVs.')

Tablas MySQL cargadas correctamente.


## 3. Vista Previa de los Datos

In [175]:
print(f"Juegos: {df_games.shape}, Steam: {df_steam.shape}, Componentes: {df_components.shape}")
print('=== JUEGOS (CSV) ===')
display(df_games.head())
print('\n=== STEAM HW SURVEY (CSV) ===')
display(df_steam.head())
print('\n=== COMPONENTES (MySQL) ===')
display(df_components)


Juegos: (15, 7), Steam: (20, 5), Componentes: (20, 4)
=== JUEGOS (CSV) ===


,game_name,cpu,ram,gpu,storage,os,target_performance
0,Cyberpunk 2077,Core i5-12400F o Ryzen 5 5600X,16 GB,RTX 3060 Ti o RX 6700 XT,70 GB SSD,Windows 10 64-bit,1080p 60fps Alta
1,Grand Theft Auto V,Core i5-6600K o Ryzen 5 1500X,8 GB,GTX 1060 6GB o RX 580 8GB,72 GB,Windows 10 64-bit,1080p 60fps Alta
2,Valorant,Core i5-9400F o Ryzen 5 2600X,8 GB,GTX 1050 Ti o RX 560,30 GB,Windows 10 64-bit,1080p 144fps Competitivo
3,Fortnite,Core i5-8400 o Ryzen 5 3600,16 GB,GTX 1080 o RX 5700,80 GB SSD,Windows 10 64-bit,1080p 60fps Alta
4,Hogwarts Legacy,Core i5-12400F o Ryzen 5 5600X,16 GB,RTX 3060 Ti o RX 6700 XT,85 GB SSD,Windows 10 64-bit,1080p 60fps Alta



=== STEAM HW SURVEY (CSV) ===


,date,category,name,change,percentage
0,2026-05-01,Video Card Description,Other,0.0016,0.0922
1,2026-05-01,Video Card Description,NVIDIA GeForce RTX 3060,-0.0014,0.0385
2,2026-05-01,Video Card Description,NVIDIA GeForce RTX 4060 Laptop GPU,-0.0001,0.0377
3,2026-05-01,Video Card Description,NVIDIA GeForce RTX 4060,-0.0031,0.0355
4,2026-05-01,Video Card Description,NVIDIA GeForce RTX 3050,0.0006,0.0310



=== COMPONENTES (MySQL) ===


,id,name,categoria,tier_name
0,1,NVIDIA GeForce GTX 1650,GPU,Gama Baja
1,5,Intel Core i3-12100F,CPU,Gama Baja
2,9,Crucial DDR4 8GB 3200MHz,RAM,Gama Baja
3,13,SSD Crucial BX500 480GB SATA,Storage,Gama Baja
4,18,Fuente Cooler Master MWE 500W,Power Supply,Gama Baja
5,20,Gabinete Cougar MX330-X,Case,Gama Baja
6,2,NVIDIA GeForce RTX 3060,GPU,Gama Media
7,4,AMD Radeon RX 6600,GPU,Gama Media
8,6,Intel Core i5-13400F,CPU,Gama Media
9,7,AMD Ryzen 5 5600X,CPU,Gama Media


---
## 4. Análisis 1 — ¿Qué exige el mercado de juegos?
Analizamos los requisitos de los 15 juegos más populares agrupados por calidad objetivo (`target_performance`).

> 💡 **Nota sobre la "Calidad Objetivo" (Valor de Negocio):**
> Los requisitos "Recomendados" oficiales suelen ser ambiguos. Durante la fase ETL, se agregó esta métrica inferida para darle sentido real a los componentes basándose en:
> 1. **El nivel del Hardware:** Si un juego pide RTX 3060 / Ryzen 5, el estándar de la industria indica que el objetivo es jugar a **1080p a 60 FPS en calidad Alta**. Si pide una RTX 3080/4070 (como Baldur's Gate 3), el objetivo salta a **1440p/4K en Ultra**.
> 2. **El género del juego (AAA vs eSports):** Juegos competitivos muy optimizados (Valorant, CS:GO2, LoL, Apex) no apuntan a 60 FPS en sus requisitos recomendados, sino a **144 FPS** para aprovechar monitores de alta tasa de refresco. A estos se les asignó `1080p 144fps Competitivo`.

Esto permite traducir nombres técnicos de hardware en **experiencias de usuario reales**, lo cual es clave para asesorar a los clientes.

In [176]:
# Conteo de juegos por target_performance
conteo_target = df_games['target_performance'].value_counts().reset_index()
conteo_target.columns = ['Calidad Objetivo', 'Cantidad de Juegos']

fig = px.bar(
    conteo_target,
    x='Calidad Objetivo', y='Cantidad de Juegos',
    color='Calidad Objetivo',
    title='Distribución de Juegos por Calidad Objetivo',
    template='plotly_white',
    text='Cantidad de Juegos'
)
fig.update_traces(textposition='outside')
fig.show()

In [177]:
# GPUs más solicitadas por los juegos (CSV)
gpus_juegos = df_games['gpu'].str.split(' o ').explode().str.strip()
top_gpus_juegos = gpus_juegos.value_counts().head(10).reset_index()
top_gpus_juegos.columns = ['GPU', 'Juegos que la piden']

# Invertimos el orden para que la mayor quede arriba
top_gpus_juegos = top_gpus_juegos.sort_values('Juegos que la piden', ascending=True)

fig2 = px.bar(
    top_gpus_juegos,
    x='Juegos que la piden', y='GPU',
    orientation='h',
    title='Top GPUs más exigidas por los juegos (Requisitos Recomendados)',
    template='plotly_white',
    # Usamos un color sólido vibrante para que los de valor 1 no queden invisibles
    color_discrete_sequence=['#0984e3'],
    text='Juegos que la piden'
)

fig2.update_traces(textposition='outside')
fig2.show()

In [178]:
# RAM más solicitada
ram_dist = df_games['ram'].value_counts().reset_index()
ram_dist.columns = ['RAM Requerida', 'Cantidad']

fig3 = px.pie(
    ram_dist,
    names='RAM Requerida', values='Cantidad',
    title='Distribución de RAM requerida en los 15 juegos',
    hole=0.4,
    template='plotly_white'
)
fig3.show()

### 💡 Conclusión
La industria se divide en dos: los juegos eSports priorizan alta fluidez (144fps) y gráficos más ligeros, mientras que los juegos AAA exigen gran potencia gráfica para mantener 60fps en calidad alta. La mayoría exige hardware de gama media como estándar mínimo.

---
## 5. Análisis 2 — ¿Qué hardware usa el mundo? (Steam HW Survey)

In [179]:
# Hardware popular según Steam (CSV directo)
df_steam_clean = df_steam[df_steam['name'] != 'Other'].copy()
df_steam_clean['porcentaje'] = (df_steam_clean['percentage'] * 100).round(2)
df_steam_clean['tipo'] = df_steam_clean['category'].apply(
    lambda x: 'GPU' if 'Video Card' in x else 'RAM'
)

# Separar GPUs y RAM
df_gpus = df_steam_clean[df_steam_clean['tipo'] == 'GPU'].sort_values('porcentaje', ascending=True)
df_ram  = df_steam_clean[df_steam_clean['tipo'] == 'RAM'].sort_values('porcentaje', ascending=True)

# Gráfico GPUs
fig_gpu = px.bar(
    df_gpus,
    x='porcentaje', y='name',
    orientation='h',
    title='GPUs más populares en Steam (% usuarios globales — Mayo 2026)',
    template='plotly_white',
    color='porcentaje',
    color_continuous_scale='Blues',
    labels={'porcentaje': '% de usuarios', 'name': 'GPU'},
    text=df_gpus['porcentaje'].apply(lambda x: f'{x:.2f}%')
)
fig_gpu.update_traces(textposition='outside')
fig_gpu.update_layout(
    coloraxis_showscale=False,
    margin=dict(r=80, l=0, t=50, b=0),
    xaxis=dict(range=[0, df_gpus['porcentaje'].max() * 1.3])
)
fig_gpu.show()

# Gráfico RAM
fig_ram = px.bar(
    df_ram,
    x='porcentaje', y='name',
    orientation='h',
    title='Distribución de RAM en Steam (% usuarios globales — Mayo 2026)',
    template='plotly_white',
    color='porcentaje',
    color_continuous_scale='Greens',
    labels={'porcentaje': '% de usuarios', 'name': 'RAM'},
    text=df_ram['porcentaje'].apply(lambda x: f'{x:.2f}%')
)
fig_ram.update_traces(textposition='outside')
fig_ram.show()


### 💡 Conclusión
El mercado es pragmático y valora el costo-beneficio. La GPU dominante indiscutible es la RTX 3060, perfecta para jugar a 1080p sin gastar de más. Esto define el "estándar" al que los desarrolladores apuntan.

---
## 6. Análisis 3 — Precios de componentes (API eBay)
¿Cuánto cuesta actualizar cada componente hoy en el mercado?

In [180]:
# Precios: consultamos usando JOIN a las tablas correspondientes
if conn:
    query = '''
    SELECT c.name as componente, m.price_clp as precio_clp
    FROM market_prices_external m
    JOIN component c ON m.component_id = c.id
    '''
    df_prices_fix = pd.read_sql(query, conn)

fig_precios = px.bar(
    df_prices_fix.sort_values('precio_clp', ascending=False),
    x='componente', y='precio_clp',
    title='Precio de Componentes en el Mercado (CLP) — Fuente: eBay API',
    template='plotly_white',
    color='precio_clp',
    color_continuous_scale='Reds',
    labels={'precio_clp': 'Precio (CLP)', 'componente': 'Componente'},
    text=df_prices_fix.sort_values('precio_clp', ascending=False)['precio_clp'].apply(lambda x: f'${x:,.0f}')
)
fig_precios.update_layout(
    height=500,
    yaxis=dict(range=[0, df_prices_fix['precio_clp'].max() * 1.2])
)
fig_precios.update_traces(textposition='outside')
fig_precios.update_xaxes(tickangle=30)
fig_precios.show()


### 💡 Conclusión
Los precios del mercado demuestran que la Tarjeta Gráfica (GPU) es el componente crítico que define el presupuesto. Es la pieza más cara y la que más impacto tiene en la experiencia de juego final.

---
## 7. Análisis 4 — Cruce Principal: ¿Cuánto cuesta armar cada Build?
Comparamos el costo total de cada build pre-armada según los precios de eBay.

In [181]:
# Costo TOTAL de builds usando la BD (todos los componentes por build)
if conn:
    query = """
    SELECT 
        bt.template_name AS build,
        c.name           AS componente,
        c.categoria,
        m.price_clp      AS precio_clp
    FROM build_templates bt
    JOIN build_components  bc ON bt.id         = bc.build_templates_id
    JOIN component         c  ON bc.component_id = c.id
    JOIN market_prices_external m ON c.id      = m.component_id
    """
    df_build_full = pd.read_sql(query, conn)
else:
    df_build_full = pd.DataFrame()

if not df_build_full.empty:
    # Calcular totales por build para el eje Y
    totales = df_build_full.groupby('build')['precio_clp'].sum().reset_index()
    totales.columns = ['build', 'total']

    # Gráfico de barras apiladas por categoría
    fig_builds = px.bar(
        df_build_full,
        x='build', y='precio_clp',
        color='categoria',
        title='Costo Total por Build — desglose por categoría (CLP)',
        template='plotly_white',
        labels={'precio_clp': 'Precio (CLP)', 'build': 'Build', 'categoria': 'Categoría'},
        barmode='stack',
        # Sin text en las barras apiladas — usamos anotaciones separadas
    )

    # Quitar texto de las barras individuales (demasiado pequeñas para algunas)
    fig_builds.update_traces(
        hovertemplate='<b>%{x}</b><br>%{data.name}: $%{y:,.0f}<extra></extra>'
    )

    # Anotar el TOTAL sobre cada barra apilada
    for _, row in totales.iterrows():
        fig_builds.add_annotation(
            x=row['build'],
            y=row['total'],
            text=f"<b>${row['total']:,.0f}</b>",
            showarrow=False,
            yanchor='bottom',
            yshift=6,
            font=dict(size=13, color='#333333')
        )

    fig_builds.update_layout(
        yaxis_tickprefix='$',
        yaxis_tickformat=',',
        legend_title_text='Categoría',
        height=500,
        uniformtext_minsize=0,
        uniformtext_mode=None
    )
    fig_builds.show()

    # Tabla resumen con desglose por categoría
    df_pivot = df_build_full.pivot_table(
        index='build', columns='categoria', values='precio_clp', aggfunc='sum', fill_value=0
    )
    df_pivot['TOTAL'] = df_pivot.sum(axis=1)
    df_pivot = df_pivot.sort_values('TOTAL')
    # Formatear como CLP
    display(df_pivot.map(lambda x: f"${x:,.0f}"))
else:
    print("Sin conexión a MySQL. No se puede calcular el costo total de builds.")

categoria,CPU,Case,GPU,Power Supply,RAM,Storage,TOTAL
build,,,,,,,
PC Gamer Entrada Economico,"$85,000","$30,000","$110,000","$28,000","$18,000","$30,000","$301,000"
PC Gamer Ultra 1080p,"$125,000","$75,000","$290,000","$50,000","$35,000","$95,000","$670,000"
PC Streamer Enthusiast 4K,"$380,000","$180,000","$560,000","$145,000","$120,000","$185,000","$1,570,000"


### 💡 Conclusión
Armar un PC de entrada es económico (aprox. $300k), pero saltar a la gama media recomendada para juegos AAA duplica el valor. Los setups "entusiastas" 4K son un nicho premium que supera los $1.5 millones.

---
## 8. Análisis 5 — ¿Qué juegos puede correr el PC Promedio de Steam?

In [182]:
matches = df_builds[df_builds['build_name'] == 'El PC Promedio de Steam']
if matches.empty:
    print("⚠️ No se encontró 'El PC Promedio de Steam' en el CSV")
    print("Builds disponibles:", df_builds['build_name'].tolist())
else:
    pc_promedio = matches.iloc[0]
# --- Sistema de tiers por serie de componente ---
def cpu_tier(cpu_str):
    cpu = str(cpu_str).lower()
    if any(x in cpu for x in ['i9', 'ryzen 9']): return 4
    if any(x in cpu for x in ['i7', 'ryzen 7']): return 3
    if any(x in cpu for x in ['i5', 'ryzen 5']): return 2
    if any(x in cpu for x in ['i3', 'ryzen 3']): return 1
    return 2

def gpu_tier(gpu_str):
    gpu = str(gpu_str).lower()
    if any(x in gpu for x in ['4080', '4090', '3090', '7900', '4070 ti', 'rx 6900']): return 4
    if any(x in gpu for x in ['4070', '3080', '3070', '6800', '6700', 'rx 6750']):    return 3
    if any(x in gpu for x in ['3060', '3050', '4060', '6600', '6500', '2060',
                               '1070', '1080', 'rx 580', 'rx 570', '960', '970', '980']): return 2
    return 1  # GTX 1650, 1060, HD4000, etc.

def ram_gb(ram_str):
    try:
        return int(str(ram_str).replace('GB','').strip().split()[0])
    except:
        return 8

tier_pc_cpu = cpu_tier(pc_promedio['cpu'])
tier_pc_ram = ram_gb(pc_promedio['ram'])
tier_pc_gpu = gpu_tier(pc_promedio['gpu'])

print(f"PC Promedio:")
print(f"  CPU : {pc_promedio['cpu'].split(' o ')[0].strip()}  (tier {tier_pc_cpu})")
print(f"  GPU : {pc_promedio['gpu'].split(' o ')[0].strip()}  (tier {tier_pc_gpu})")
print(f"  RAM : {pc_promedio['ram']}")

def puede_correr(row):
    # Columnas del CSV: game_name, cpu, gpu, ram, storage, os, target_performance
    ram_juego     = ram_gb(row['ram'])
    cpu_juego_str = str(row['cpu']).split(' o ')[0].strip()
    gpu_juego_str = str(row['gpu']).split(' o ')[0].strip()

    tier_cpu_juego = cpu_tier(cpu_juego_str)
    tier_gpu_juego = gpu_tier(gpu_juego_str)

    problemas = []
    if ram_juego > tier_pc_ram:
        problemas.append('RAM insuf.')
    if tier_cpu_juego > tier_pc_cpu:
        problemas.append('CPU insuf.')
    if tier_gpu_juego > tier_pc_gpu:
        problemas.append('GPU insuf.')

    if problemas:
        return '❌ ' + ' + '.join(problemas)
    elif any(x in str(row.get('game_name', '')).lower()
             for x in ['valorant', 'league', 'cs', 'overwatch', 'apex', 'minecraft', 'fortnite']):
        return '✅ Sí (eSports)'
    else:
        return '⚠️ Sí (calidad media)'

df_games['estado_pc_promedio'] = df_games.apply(puede_correr, axis=1)

# Gráfico de torta
conteo_estado = df_games['estado_pc_promedio'].value_counts().reset_index()
conteo_estado.columns = ['Estado', 'Cantidad de Juegos']

fig_pc = px.pie(
    conteo_estado,
    names='Estado', values='Cantidad de Juegos',
    hole=0.4,
    template='plotly_white',
)
fig_pc.update_layout(
    title=dict(
        text='¿Puede el PC Promedio de Steam correr los 15 juegos?<br><sup>Specs: RTX 3060 (tier 2) / i5-10400F (tier 2) / 16 GB RAM</sup>',
        x=0.5, xanchor='center'
    ),
    margin=dict(t=90)
)
fig_pc.show()

# Tabla resumen
display(df_games[['game_name','cpu','gpu','ram','estado_pc_promedio']].sort_values('estado_pc_promedio'))

PC Promedio:
  CPU : Core i5-10400F  (tier 2)
  GPU : RTX 3060  (tier 2)
  RAM : 16 GB


,game_name,cpu,gpu,ram,estado_pc_promedio
0,Cyberpunk 2077,Core i5-12400F o Ryzen 5 5600X,RTX 3060 Ti o RX 6700 XT,16 GB,⚠️ Sí (calidad media)
1,Grand Theft Auto V,Core i5-6600K o Ryzen 5 1500X,GTX 1060 6GB o RX 580 8GB,8 GB,⚠️ Sí (calidad media)
4,Hogwarts Legacy,Core i5-12400F o Ryzen 5 5600X,RTX 3060 Ti o RX 6700 XT,16 GB,⚠️ Sí (calidad media)
12,Counter-Strike 2,Core i5-9400F o Ryzen 5 2600X,GTX 1060 6GB o RX 580,16 GB,⚠️ Sí (calidad media)
2,Valorant,Core i5-9400F o Ryzen 5 2600X,GTX 1050 Ti o RX 560,8 GB,✅ Sí (eSports)
3,Fortnite,Core i5-8400 o Ryzen 5 3600,GTX 1080 o RX 5700,16 GB,✅ Sí (eSports)
6,Minecraft,Core i5-8400 o Ryzen 5 2600,GTX 1060 o RX 580,8 GB,✅ Sí (eSports)
7,League of Legends,Core i5-3300 o Ryzen 3 1300X,GTX 960 o RX 470,8 GB,✅ Sí (eSports)
11,Apex Legends,Core i5-8600K o Ryzen 5 3600,RTX 2060 o RX 5700,16 GB,✅ Sí (eSports)
5,Red Dead Redemption 2,Core i7-9700K o Ryzen 7 3700X,RTX 2080 o RX 5700 XT,16 GB,❌ CPU insuf.


### 💡 Conclusión
El "PC Promedio" global es una máquina bastante capaz. Puede correr todos los juegos de eSports y la gran mayoría de los AAA modernos en 1080p, confirmando por qué configuraciones como la RTX 3060 son líderes del mercado.

---
## 9. Análisis 6 — Componentes por Gama (Tiers)
Distribución del catálogo de componentes por nivel de rendimiento.

In [183]:
# Distribución del catálogo por categoría y gama (desde MySQL)
if conn and not df_components.empty:
    df_cat = df_components.copy()
    df_cat.columns = ['id', 'componente', 'categoria', 'gama']
else:
    # Fallback manual (20 componentes)
    df_cat = pd.DataFrame({
        'componente': [
            'NVIDIA GeForce GTX 1650', 'NVIDIA GeForce RTX 3060', 'NVIDIA GeForce RTX 4070', 'AMD Radeon RX 6600',
            'Intel Core i3-12100F', 'Intel Core i5-13400F', 'AMD Ryzen 5 5600X', 'AMD Ryzen 7 7800X3D',
            'Crucial DDR4 8GB', 'Kingston Fury DDR4 16GB', 'Corsair Vengeance DDR5 32GB',
            'SSD Crucial BX500 480GB', 'SSD Kingston NV2 1TB', 'SSD Samsung 990 Pro 2TB',
            'Fuente Cooler Master 500W', 'Fuente MSI MAG 650W', 'Fuente Corsair RM1000x 1000W',
            'Gabinete Cougar MX330-X', 'Gabinete MSI Forge 112R', 'Gabinete Lian Li O11'
        ],
        'categoria': [
            'GPU','GPU','GPU','GPU',
            'CPU','CPU','CPU','CPU',
            'RAM','RAM','RAM',
            'Storage','Storage','Storage',
            'Power Supply','Power Supply','Power Supply',
            'Case','Case','Case'
        ],
        'gama': [
            'Gama Baja','Gama Media','Gama Alta','Gama Media',
            'Gama Baja','Gama Media','Gama Media','Gama Alta',
            'Gama Baja','Gama Media','Gama Alta',
            'Gama Baja','Gama Media','Gama Alta',
            'Gama Baja','Gama Media','Gama Alta',
            'Gama Baja','Gama Media','Gama Alta'
        ]
    })

tier_dist = df_cat.groupby(['categoria', 'gama'], as_index=False).size()
tier_dist.columns = ['categoria', 'gama', 'cantidad']

# Orden visual de gamas
orden_gama = ['Gama Baja', 'Gama Media', 'Gama Alta']
tier_dist['gama'] = pd.Categorical(tier_dist['gama'], categories=orden_gama, ordered=True)
tier_dist = tier_dist.sort_values(['categoria', 'gama'])

fig8 = px.bar(
    tier_dist,
    x='categoria', y='cantidad',
    color='gama',
    barmode='group',
    title='Catálogo de componentes: 20 modelos distribuidos en 3 gamas y 6 categorías',
    template='plotly_white',
    labels={'categoria': 'Tipo de Componente', 'cantidad': 'Cantidad', 'gama': 'Gama'},
    color_discrete_map={
        'Gama Baja':  '#74b9ff',
        'Gama Media': '#0984e3',
        'Gama Alta':  '#2d3436'
    },
    text='cantidad'
)
fig8.update_traces(textposition='outside')
fig8.update_layout(
    yaxis=dict(range=[0, tier_dist['cantidad'].max() + 1]),
    height=450
)
fig8.show()

print(f"\nTotal componentes en catálogo: {len(df_cat)}")
print(tier_dist.pivot_table(index='categoria', columns='gama', values='cantidad', fill_value=0).to_string())


Total componentes en catálogo: 20
gama          Gama Baja  Gama Media  Gama Alta
categoria                                     
CPU                 1.0         2.0        1.0
Case                1.0         1.0        1.0
GPU                 1.0         2.0        1.0
Power Supply        1.0         1.0        1.0
RAM                 1.0         1.0        1.0
Storage             1.0         1.0        1.0


### 💡 Conclusión
El catálogo tiene una oferta escalonada ideal. Esto permite construir opciones viables para todo tipo de cliente: desde el gamer casual con presupuesto ajustado, hasta el entusiasta que busca rendimiento extremo sin importar el precio.

---
## 10. Conclusiones y Valor del Sistema

### Hallazgos principales:

1. **Demanda del mercado (Fuente: Kaggle):** La mayoría de los juegos AAA requieren una GPU de Gama Media (RTX 3060 / RX 6600) para correr a 1080p/60fps en calidad alta.

2. **El PC promedio global (Fuente: Steam HW Survey):** La RTX 3060 es la GPU más popular, y el 42% de usuarios ya tiene 16GB de RAM — lo que indica que el mercado está migrando hacia ese estándar.

3. **Precios en tiempo real (Fuente: eBay Browse API v1):** La GPU es el componente más caro y tiene mayor impacto en la capacidad de juego. Representa entre el 40-60% del costo total de un build.

4. **Brecha de actualización:** Los juegos eSports (Valorant, Counter-Strike 2, LoL) corren perfectamente en hardware de gama baja/media, pero los AAA (Cyberpunk, Hogwarts Legacy) requieren una inversión real.

5. **Build recomendada:** El "PC Gamer Ultra 1080p" ofrece la mejor relación costo/beneficio para cubrir la mayoría de los juegos de la muestra.

### Valor del Sistema (Business Value)
Este proyecto demuestra cómo cruzar múltiples fuentes de datos distribuidas (archivos estáticos, APIs REST en tiempo real y bases de datos relacionales) permite generar inteligencia de mercado procesable.
El sistema empodera al usuario final para tomar decisiones de compra informadas, identificando qué componente priorizar según su juego objetivo y presupuesto disponible.


In [184]:
# Cerrar conexión MySQL
if conn:
    conn.close()
    print('Conexión MySQL cerrada.')

Conexión MySQL cerrada.
